# Article features

How returnable the items themselves are: past-only return rates for the article, its size, and its product group, computed on order lines and then rolled up to the order.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from src.data import INTERIM, load_orders, save_features
from src.features import past_rate

## Lines

**Line-level target and a composite key.** `size_key` concatenates article and size, so an item's return rate can be tracked per size — a shirt that runs small returns at a different rate in M than in XL.

In [2]:
lines = load_orders("orders_train")
lines["returned"] = (lines.returnQuantity > 0).astype(int)
lines["size_key"] = lines.articleID.astype(str) + "_" + lines.sizeCode.astype(str)
lines.shape

(2325165, 17)

## Past-only rates

**Past-only rates at three grains:** article, article×size, and product group. Same `past_rate` helper as customer history, so the same same-day protection applies. Smoothing pulls sparse keys toward the population mean instead of letting a single observation produce a 0.0 or 1.0 rate.

In [3]:
for key, name in [("articleID", "article"), ("size_key", "artsize"), ("productGroup", "group")]:
    rate, n = past_rate(lines, key, "orderDate", "returned")
    lines[f"{name}_past_rate"] = rate
    lines[f"{name}_prior_n"] = n
lines[[c for c in lines.columns if c.endswith("_past_rate")]].describe().round(4)

,article_past_rate,artsize_past_rate,group_past_rate
count,2.325165e+06,2.325165e+06,2.324814e+06
mean,5.376000e-01,5.245000e-01,5.267000e-01
std,1.118000e-01,9.440000e-02,8.630000e-02
min,6.490000e-02,6.490000e-02,9.620000e-02
25%,4.815000e-01,4.861000e-01,4.601000e-01
50%,5.486000e-01,5.346000e-01,5.140000e-01
75%,6.106000e-01,5.824000e-01,5.982000e-01
max,8.747000e-01,8.542000e-01,6.987000e-01


## Roll up

**Roll line-level rates up to the order.** Both mean and max are kept: the max captures *the riskiest single item in the basket*, which is often more informative than the average, since one notorious item can sink an otherwise safe order.

In [4]:
article = lines.groupby("orderID").agg(
    article_rate_mean=("article_past_rate", "mean"),
    article_rate_max=("article_past_rate", "max"),
    article_prior_n_mean=("article_prior_n", "mean"),
    artsize_rate_mean=("artsize_past_rate", "mean"),
    artsize_rate_max=("artsize_past_rate", "max"),
    group_rate_mean=("group_past_rate", "mean")).reset_index()
save_features(article, "article", keys=("orderID",))
article.shape

(738698, 7)